In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. SETUP & DRIVE MOUNT
from google.colab import drive
import json
import re
import os
import pandas as pd
import random
from sklearn.model_selection import train_test_split

# Mount Drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/685 final project"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "/content/drive/MyDrive/685_final_project"

INPUT_FILE = "data/dataset_v4_final.json"
OUTPUT_DIR = os.path.join(BASE_PATH, "data")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(f"📂 Working Directory: {BASE_PATH}")

# --- MAIN SPLIT LOGIC ---
full_input_path = os.path.join(BASE_PATH, INPUT_FILE)

if not os.path.exists(full_input_path):
    full_input_path = INPUT_FILE

print(f"📖 Loading {full_input_path}...")
with open(full_input_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 1. Simple Random Shuffle
random.seed(42)
random.shuffle(data)
total = len(data)

# 2. Split Logic: 80 / 10 / 10
# First, peel off 10% for Test
train_val, test_set = train_test_split(data, test_size=0.10, random_state=42)

# Next, peel off 10% (of the TOTAL) for Val.
# The remaining pool (train_val) is 90% of total.
# We need 10% of total, which is (10/90) = 1/9th of the remaining pool.
val_size_adjusted = 10 / 90
train_set, val_set = train_test_split(train_val, test_size=val_size_adjusted, random_state=42)

print(f"Split Results:")
print(f"   Train: {len(train_set)} ({(len(train_set)/total)*100:.1f}%)")
print(f"   Val:   {len(val_set)}   ({(len(val_set)/total)*100:.1f}%)")
print(f"   Test:  {len(test_set)}  ({(len(test_set)/total)*100:.1f}%)")

# 3. Generate Task Files
def save_task_files(dataset, split_name):
    lines = []

    for entry in dataset:
        # Construct Control Tags
        tags = entry.get('control_tags', '')
        if not tags:
             tags = f"<AGE={entry.get('age_group')}> <THEME={entry.get('theme')}> <LENGTH={entry.get('length')}>"

        story = entry['story']
        prompt = f"[CTRL] {tags} [INST] Write a fairy tale that follows the CTRL conditions.\n"
        completion = story
        lines.append({"prompt": prompt, "completion": completion})

    # Save to JSONL
    p_path = os.path.join(OUTPUT_DIR, f"instruction_{split_name}.jsonl")

    with open(p_path, 'w', encoding='utf-8') as f:
        for l in lines: f.write(json.dumps(l, ensure_ascii=False) + "\n")

save_task_files(train_set, "train")
save_task_files(val_set, "val")
save_task_files(test_set, "test")

print(f"Data Prep Complete. Files saved to: {OUTPUT_DIR}")

In [ ]:
!pip -q install -U bitsandbytes accelerate transformers peft datasets


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import os
import gc
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorWithPadding
)


from google.colab import drive
drive.mount('/content/drive')

# Configuration
BASE_PATH = "/content/drive/MyDrive/685 final project"
if not os.path.exists(BASE_PATH): BASE_PATH = "/content/drive/MyDrive/685_final_project"

MODEL_NAME = "issai/LLama-3.1-KazLLM-1.0-8B"
OUTPUT_DIR = os.path.join(BASE_PATH, "models/instruction_tuned_kazllm")


MAX_SEQ_LENGTH = 2048
print(f"📉 Reduced Context Length to {MAX_SEQ_LENGTH} to prevent OOM.")

# Load Data
dataset = load_dataset("json", data_files={
    "train": os.path.join(BASE_PATH, "data/instruction_train.jsonl"),
    "validation": os.path.join(BASE_PATH, "data/instruction_val.jsonl")
})

# 2. MODEL
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    max_memory={0: "20GiB", "cpu": "48GiB"}
)
model.config.use_cache = False

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
model = prepare_model_for_kbit_training(model)

#  non-reentrant checkpointing to reduce memory
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,token=True,  use_fast = True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
model.config.pad_token_id = tokenizer.eos_token_id



# 3. TOKENIZE
print(f"Tokenizing for {MAX_SEQ_LENGTH} length...")
def tokenize_prompt_completion(examples):
    prompts = examples["prompt"]
    completions = examples["completion"]

    prompt_tok = tokenizer(prompts, add_special_tokens=False)
    comp_tok   = tokenizer(completions, add_special_tokens=False)

    input_ids_list, labels_list, attn_list = [], [], []

    for p_ids, c_ids in zip(prompt_tok["input_ids"], comp_tok["input_ids"]):
        # EOS at end of completion
        c_ids = c_ids + [tokenizer.eos_token_id]

        input_ids = p_ids + c_ids
        labels    = ([-100] * len(p_ids)) + c_ids
        attn_mask = [1] * len(input_ids)

        # Truncate: prefer keeping completion tokens
        if len(input_ids) > MAX_SEQ_LENGTH:
            overflow = len(input_ids) - MAX_SEQ_LENGTH

            # cut from the start of prompt first
            keep_p = max(len(p_ids) - overflow, 0)
            p_ids = p_ids[-keep_p:]

            input_ids = p_ids + c_ids
            labels    = ([-100] * len(p_ids)) + c_ids
            attn_mask = [1] * len(input_ids)

            # if still too long, cut the tail
            input_ids = input_ids[:MAX_SEQ_LENGTH]
            labels    = labels[:MAX_SEQ_LENGTH]
            attn_mask = attn_mask[:MAX_SEQ_LENGTH]

        input_ids_list.append(input_ids)
        labels_list.append(labels)
        attn_list.append(attn_mask)

    return {
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attn_list
    }
tokenized_ds = dataset.map(
    tokenize_prompt_completion,
    batched=True,
    remove_columns=["prompt", "completion"]
)
ex = tokenized_ds["train"][0]
masked = sum(1 for x in ex["labels"] if x == -100)
unmasked = sum(1 for x in ex["labels"] if x != -100)
print("masked(prompt) =", masked, "unmasked(completion) =", unmasked)
assert masked > 0 and unmasked > 0

assert len(ex["input_ids"]) == len(ex["labels"]) == len(ex["attention_mask"])

# --------------------
# COLLATOR: pad labels with -100
# --------------------

def completion_only_collator(features):
    pad_id = tokenizer.pad_token_id

    max_len = max(len(f["input_ids"]) for f in features)

    input_ids, attention_mask, labels = [], [], []
    for f in features:
        ids = f["input_ids"]
        attn = f["attention_mask"]
        lab = f["labels"]

        pad_len = max_len - len(ids)

        input_ids.append(ids + [pad_id] * pad_len)
        attention_mask.append(attn + [0] * pad_len)
        labels.append(lab + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

# 4. TRAIN
peft_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)
model.add_adapter(peft_config)

# --- CRITICAL MEMORY CLEAR BEFORE TRAIN ---
gc.collect()
torch.cuda.empty_cache()

trainer = Trainer(
    model=model,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=1e-4,
        num_train_epochs=3,
        logging_steps=1,
        eval_strategy="epoch",

        save_strategy="no",
        report_to="none",
        bf16=True,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        eval_accumulation_steps=1,
        prediction_loss_only=True,
        ),
    data_collator=completion_only_collator
)

print("Attempting Training with 8-bit Optimizer + 2048 Context...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model Saved to: {OUTPUT_DIR}")